# Allscripts Touchworks — Note Hydration

Populates `_exponent.omop_tw.note` from Allscripts Touchworks clinical result/report notes.

## Source Tables
- `_bronze_allscripts_tw_works.dbo_order_activity_header` — note metadata (patient, encounter, visit, date)
- `_bronze_allscripts_tw_works.dbo_order_result_mapper` — links order activity header to result activity header
- `_bronze_allscripts_tw_works.dbo_result_text` — note body text

## Join Chain
`dbo_order_activity_header.ID` → `dbo_order_result_mapper.OrderActivityHeaderID`
→ `dbo_order_result_mapper.ResultActivityHeaderID` = `dbo_result_text.ResultID`

## Pipeline
1. `silver_note` — staged temp view, full OMOP field set
2. MERGE → `omop_silver.note`
3. INSERT → `omop_mapping.source_to_note`
4. `gold` — resolves surrogate IDs and FK references
5. MERGE → `omop_tw.note`

## Dependencies
- `omop_mapping.source_to_person` must be populated for allscripts_tw
- `omop_mapping.source_to_visit_occurrence` must be populated for allscripts_tw

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_allscripts.note;

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_tw.note;

In [0]:
%sql
DELETE FROM _exponent.omop_silver.note
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
DELETE FROM _exponent.omop_mapping.source_to_note
WHERE source_system = 'allscripts_tw';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_note AS
SELECT
  CONCAT_WS(
    CHR(31),
    'allscripts_tw',
    'report_ahs_mtemplate',
    'ID',
    CAST(report_ahs_mtemplate.ID AS BIGINT)
  ) AS note_source_value,

  source_to_person.person_id,

  CAST(
    COALESCE(
      report_ahs_mtemplate.ClinicalDTTM,
      report_ahs_mtemplate.PerformedDTTM,
      dbo_vendor_item.PerformedDTTM,
      dbo_vendor_item.RecordedDTTM,
      report_ahs_mtemplate.CreateDTTM,
      dbo_vendor_item.CreateDTTM
    ) AS DATE
  ) AS note_date,

  COALESCE(
    report_ahs_mtemplate.ClinicalDTTM,
    report_ahs_mtemplate.PerformedDTTM,
    dbo_vendor_item.PerformedDTTM,
    dbo_vendor_item.RecordedDTTM,
    report_ahs_mtemplate.CreateDTTM,
    dbo_vendor_item.CreateDTTM
  ) AS note_datetime,

  32817 AS note_type_concept_id,
  3030653 AS note_class_concept_id,

  COALESCE(
    report_ahs_mtemplate.DocumentName,
    report_ahs_mtemplate.NoteSectionName,
    report_ahs_mtemplate.NoteFormSectionName,
    report_ahs_mtemplate.DisplayName
  ) AS note_title,

  CONCAT_WS(
    '\n',
    CONCAT('Document: ', COALESCE(report_ahs_mtemplate.DocumentName, '')),
    CONCAT('Section: ', COALESCE(report_ahs_mtemplate.NoteSectionName, '')),
    CONCAT('Form Section: ', COALESCE(report_ahs_mtemplate.NoteFormSectionName, '')),
    CONCAT(
      'Finding: ',
      COALESCE(
        report_ahs_mtemplate.medcinfinding,
        report_ahs_mtemplate.DisplayName,
        report_ahs_mtemplate.QO_DE,
        ''
      )
    ),
    CONCAT('Answer: ', COALESCE(report_ahs_mtemplate.answer, '')),
    CONCAT('Additional Text: ', COALESCE(dbo_vendor_item_extension.ExtensionValue, ''))
  ) AS note_text,

  32678 AS encoding_concept_id,
  4180186 AS language_concept_id,

  source_to_provider.provider_id,
  source_to_visit_occurrence.visit_occurrence_id,

  CASE
    WHEN dbo_visit.id IS NOT NULL
      THEN CONCAT_WS(
             CHR(31),
             'allscripts_tw',
             'dbo_visit',
             'id',
             CAST(dbo_visit.id AS BIGINT)
           )
    ELSE NULL
  END AS visit_occurrence_source_value,

  NULL AS visit_detail_id,
  NULL AS note_event_id,
  NULL AS note_event_field_concept_id,

  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw.report_ahs_mtemplate

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_vendor_item
  ON dbo_vendor_item.ID = report_ahs_mtemplate.ItemID
 AND (
      dbo_vendor_item.IsErrorFLAG <> 'Y'
      OR dbo_vendor_item.IsErrorFLAG IS NULL
 )

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_vendor_item_extension
  ON dbo_vendor_item_extension.VendorItemID = dbo_vendor_item.ID
 AND dbo_vendor_item_extension.ExtensionType = 'AddedText'

JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  ON dbo_encounter.id = CAST(
       COALESCE(
         report_ahs_mtemplate.encounterID,
         dbo_vendor_item.EncounterID
       ) AS BIGINT
     )

JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
  ON dbo_visit.id = dbo_encounter.visitid

JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(report_ahs_mtemplate.patientid AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(
         COALESCE(
           report_ahs_mtemplate.WhoDidItID,
           dbo_vendor_item.WhoDidItID
         ) AS BIGINT
       )
     )
 AND source_to_provider.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_visit',
       'id',
       CAST(dbo_visit.id AS BIGINT)
     )
 AND source_to_visit_occurrence.active_flag = TRUE

WHERE report_ahs_mtemplate.patientid IS NOT NULL
  AND COALESCE(
        report_ahs_mtemplate.ClinicalDTTM,
        report_ahs_mtemplate.PerformedDTTM,
        dbo_vendor_item.PerformedDTTM,
        dbo_vendor_item.RecordedDTTM,
        report_ahs_mtemplate.CreateDTTM,
        dbo_vendor_item.CreateDTTM
      ) IS NOT NULL
  AND CAST(
        COALESCE(
          report_ahs_mtemplate.ClinicalDTTM,
          report_ahs_mtemplate.PerformedDTTM,
          dbo_vendor_item.PerformedDTTM,
          dbo_vendor_item.RecordedDTTM,
          report_ahs_mtemplate.CreateDTTM,
          dbo_vendor_item.CreateDTTM
        ) AS DATE
      ) >= '1950-01-01'
  AND (
       report_ahs_mtemplate.answer IS NOT NULL
       AND TRIM(report_ahs_mtemplate.answer) <> ''
       OR dbo_vendor_item_extension.ExtensionValue IS NOT NULL
       AND TRIM(dbo_vendor_item_extension.ExtensionValue) <> ''
  )
  AND (
        report_ahs_mtemplate.IsErrorFlag <> 'Y'
        OR report_ahs_mtemplate.IsErrorFlag IS NULL
      );

In [0]:
%sql
MERGE INTO _exponent.omop_silver.note AS target
USING silver_note AS source
ON target.note_source_value = source.note_source_value

WHEN MATCHED AND (
     NOT (target.person_id <=> source.person_id)
  OR NOT (target.note_date <=> source.note_date)
  OR NOT (target.note_datetime <=> source.note_datetime)
  OR NOT (target.note_type_concept_id <=> source.note_type_concept_id)
  OR NOT (target.note_class_concept_id <=> source.note_class_concept_id)
  OR NOT (target.note_title <=> source.note_title)
  OR NOT (target.note_text <=> source.note_text)
  OR NOT (target.encoding_concept_id <=> source.encoding_concept_id)
  OR NOT (target.language_concept_id <=> source.language_concept_id)
  OR NOT (target.provider_id <=> source.provider_id)
  OR NOT (target.visit_occurrence_id <=> source.visit_occurrence_id)
  OR NOT (target.visit_detail_id <=> source.visit_detail_id)
  OR NOT (target.note_event_id <=> source.note_event_id)
  OR NOT (target.note_event_field_concept_id <=> source.note_event_field_concept_id)
  OR NOT (target.source_system <=> source.source_system)
)
THEN UPDATE SET
  target.person_id = source.person_id,
  target.note_date = source.note_date,
  target.note_datetime = source.note_datetime,
  target.note_type_concept_id = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title = source.note_title,
  target.note_text = source.note_text,
  target.encoding_concept_id = source.encoding_concept_id,
  target.language_concept_id = source.language_concept_id,
  target.provider_id = source.provider_id,
  target.visit_occurrence_id = source.visit_occurrence_id,
  target.visit_detail_id = source.visit_detail_id,
  target.note_event_id = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.source_system = source.source_system,
  target.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  note_source_value,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  source.note_source_value,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_note (
    source_system,
    note_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    source.source_system,
    source.note_source_value,
    TRUE                AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    NULL                AS merge_id,
    NULL                AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        note_source_value
    FROM silver_note
    WHERE note_source_value IS NOT NULL
      AND source_system = 'allscripts_tw'
) source
LEFT ANTI JOIN _exponent.omop_mapping.source_to_note target
  ON target.note_source_value = source.note_source_value
 AND target.source_system = source.source_system;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  source_to_note.note_id,

  note.person_id,

  note.note_date,
  note.note_datetime,
  note.note_type_concept_id,
  note.note_class_concept_id,
  note.note_title,
  note.note_text,
  note.encoding_concept_id,
  note.language_concept_id,
  note.provider_id,

  note.visit_occurrence_id,

  note.visit_detail_id,
  note.note_event_id,
  note.note_event_field_concept_id,
  note.note_source_value

FROM _exponent.omop_silver.note

JOIN _exponent.omop_mapping.source_to_note
  ON source_to_note.note_source_value = note.note_source_value
 AND source_to_note.source_system = 'allscripts_tw'
 AND source_to_note.active_flag = TRUE

WHERE note.source_system = 'allscripts_tw';

In [0]:
%sql
MERGE INTO _exponent.omop_tw.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);